In [ ]:
%run init_env.py

Environment initializing completed successfully.


In [2]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dataclass
class LanguageContext:
    user_language: str = "English"

@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_language = request.runtime.context.user_language
    base_prompt = "You are a helpful assistant."

    if user_language != "English":
        return f"{base_prompt} only respond in {user_language}."
    elif user_language == "English":
        return base_prompt

In [3]:
from databricks_langchain import ChatDatabricks
from langchain.agents import create_agent

agent = create_agent(
    model=ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", temperature=0, max_tokens=500),
    context_schema=LanguageContext,
    middleware=[user_language_prompt]
)

### Following steps hard codes the user selected language, which ideally, you would have prompted the user through your app ...

In [4]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Irish")
)

print(response["messages"][-1].content)

Dia dhuit, conas atá tú?


In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Spanish")
)

print(response["messages"][-1].content)

Hola, ¿cómo estás?


In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="French")
)

print(response["messages"][-1].content)

Bonjour, je vais bien, merci ! Comment puis-je vous aider aujourd'hui ?
